In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from conus_biomass import dir_info
from conus_biomass.make_figures import maps
from conus_biomass.process_outputs.generate_state_csv import calculate_total_carbon_stock

# a)

In [ ]:
def get_filtered_output(
    year,
    fdir=dir_info.dir_processed
    + "model_results/1000m_2026feb02_processed/",  # dir_info.dir_model_output,
    ftype="netcdf",
    vartype="predicted_biomass_filtered_",
):
    fname = fdir + vartype + str(year)
    if ftype == "zarr":
        ds = xr.open_zarr(fname + ".zarr")
    elif ftype == "netcdf":
        ds = xr.load_dataset(fname + ".nc")
    return ds


def clip_to_shape(da, shp):
    if shp.crs != da.rio.crs:
        shp = shp.to_crs(da.rio.crs)
        print("reprojected shape")
    clipped = da.rio.clip(shp.geometry, shp.crs, drop=True)
    return clipped

In [ ]:
usfs = pd.read_csv("figure_data/figure_2/USFS_western_stocks.csv")

In [ ]:
usfs_deltas = usfs["0"].values[1:] - usfs["0"].values[:-1]

In [ ]:
usfs_years = usfs["Unnamed: 0"].values[:-1]

In [ ]:
slope = xr.open_zarr(dir_info.dir_processed + "data_on_ref_grid/1000m/aspect.zarr")
crs = slope["spatial_ref"].crs_wkt

In [ ]:
fire_losses_west = []
years = np.arange(2005, 2023)

for year in years:
    print(year)
    biomass_forest = get_filtered_output(
        year=year, vartype="predicted_biomass_delta_burned_filtered_"
    )
    biomass_forest = biomass_forest.rio.write_crs(crs)
    biomass_forest = biomass_forest

    biomass_forest_west = clip_to_shape(
        da=biomass_forest["predicted_biomass_delta_burned"], shp=maps.SHP_WESTERN
    )
    MMT_west = calculate_total_carbon_stock(biomass_forest_west, grid_res=1000)
    fire_losses_west.append(MMT_west)

    del biomass_forest, biomass_forest_west

In [ ]:
unburned_gains_west = []

for year in years:
    print(year)
    biomass_forest = get_filtered_output(
        year=year, vartype="predicted_biomass_delta_unburned_filtered_"
    )
    biomass_forest = biomass_forest.rio.write_crs(crs)
    biomass_forest = biomass_forest

    biomass_forest_west = clip_to_shape(
        da=biomass_forest["predicted_biomass_delta_unburned"], shp=maps.SHP_WESTERN
    )
    MMT_west = calculate_total_carbon_stock(biomass_forest_west, grid_res=1000)
    unburned_gains_west.append(MMT_west)

    del biomass_forest, biomass_forest_west

In [ ]:
net_changes = np.array(unburned_gains_west) + np.array(fire_losses_west)

In [ ]:
plt.rcParams.update({"font.size": 14})
fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(16 / 2, 5))

ax = axes

ax.set_ylabel("Live AGB Change per year (Tg C/year)")
ax.spines[["right", "top"]].set_visible(False)
ax.legend(frameon=False)
ax.set_xlim([2004, 2024])
ax.axhline(y=0, linestyle=":", color="gray", linewidth=1)

# ax.axvspan(2005, 2015, color="lightgray", alpha=0.3)
# ax.axvspan(2015, 2022, color="lightgray", alpha=0.3)

ax.plot(
    years, fire_losses_west, linewidth=5, color="#f07071", label="Burn losses"
)  # , marker='o', markersize=10)
ax.plot(
    years, unburned_gains_west, linewidth=5, color="#64b9c4", label="Unburned gains"
)  # , marker='o', markersize=10)
ax.plot(
    years, net_changes, linewidth=5, color="#85a2f7", label="Net change"
)  # , marker='o', markersize=10)
ax.plot(
    usfs_years, usfs_deltas, linewidth=5, color="darkgray", label="USFS/EPA net change", zorder=0
)
plt.legend(frameon=False)

rolling_avg = pd.Series(net_changes).rolling(window=5, center=True).mean()
ax.plot(
    years,
    rolling_avg,
    linewidth=3,
    color="#85a2f7",
    label="Net change (5-yr avg)",
    linestyle="--",
    alpha=0.8,
)
rolling_avg = pd.Series(np.array(unburned_gains_west)).rolling(window=5, center=True).mean()
ax.plot(
    years,
    rolling_avg,
    linewidth=3,
    color="#64b9c4",
    label="Net change (5-yr avg)",
    linestyle="--",
    alpha=0.8,
)
rolling_avg = pd.Series(np.array(fire_losses_west)).rolling(window=5, center=True).mean()
ax.plot(
    years,
    rolling_avg,
    linewidth=3,
    color="#f07071",
    label="Net change (5-yr avg)",
    linestyle="--",
    alpha=0.8,
)

plt.tight_layout()
plt.savefig("Annual_changes.png", dpi=200)
plt.xlim([2005, 2023])
# ax.plot(years,np.array(fire_losses_west)+
#        np.array(unburned_gains_west), linewidth=5, color='C1', marker='o', markersize=10)

In [ ]:
plt.rcParams.update({"font.size": 14})
fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(16 / 2, 5))

ax = axes

ax.set_ylabel("Live AGB Change per year (Tg C/year)")
ax.spines[["right", "top"]].set_visible(False)
ax.legend(frameon=False)
ax.set_xlim([2004, 2024])
ax.axhline(y=0, linestyle=":", color="gray", linewidth=1)

# ax.axvspan(2005, 2015, color="lightgray", alpha=0.3)
# ax.axvspan(2015, 2022, color="lightgray", alpha=0.3)

ax.plot(
    years + 1, np.cumsum(fire_losses_west), linewidth=5, color="#f07071", label="Burn losses"
)  # , marker='o', markersize=10)
ax.plot(
    years + 1, np.cumsum(unburned_gains_west), linewidth=5, color="#64b9c4", label="Unburned gains"
)  # , marker='o', markersize=10)
ax.plot(
    years + 1, np.cumsum(net_changes), linewidth=5, color="#85a2f7", label="Net change"
)  # , marker='o', markersize=10)
ax.plot(
    usfs_years[15:],
    np.cumsum(usfs_deltas[15:]),
    linewidth=5,
    color="darkgray",
    label="USFS/EPA net change",
    zorder=0,
)

plt.tight_layout()
plt.savefig("Annual_changes.png", dpi=200)
plt.xlim([2005, 2023])
# ax.plot(years,np.array(fire_losses_west)+
#        np.array(unburned_gains_west), linewidth=5, color='C1', marker='o', markersize=10)

# Panel b)

In [ ]:
slope = xr.open_zarr(dir_info.dir_processed + "data_on_ref_grid/1000m/aspect.zarr")
crs = slope["spatial_ref"].crs_wkt

In [ ]:
inputs = xr.open_dataset(dir_info.dir_processed + "data_on_ref_grid/1000m/all_variables.nc")

In [ ]:
dir_data = dir_info.dir_model_output[:-1] + "_processed/"

In [ ]:
slope = xr.open_zarr(dir_info.dir_processed + "data_on_ref_grid/1000m/aspect.zarr")
crs = slope["spatial_ref"].crs_wkt
western_states = maps.SHP_WESTERN.to_crs(crs)

In [ ]:
def get_filtered_output(
    year,
    fdir=dir_data,
    ftype="nc",
    vartype="predicted_biomass_filtered_",
):
    fname = fdir + vartype + str(year)
    if ftype == "zarr":
        ds = xr.open_zarr(fname + ".zarr")
    elif ftype == "nc":
        ds = xr.load_dataset(fname + "_0000.nc")
    return ds


ds_end = get_filtered_output(year=2022)
ds_start = get_filtered_output(year=2005)
delta_biomass = ds_end["predicted_biomass"] - ds_start["predicted_biomass"]

In [ ]:
inputs = xr.open_dataset(dir_info.dir_processed + "data_on_ref_grid/1000m/all_variables.nc")
inputs = inputs.rio.write_crs(crs)

In [ ]:
forest_area = inputs["forest_remaining_forest"].sel(year=2005).rio.write_crs(crs)

In [ ]:
burned = inputs["years_after_fire"].sel(year=2022) <= 17

In [ ]:
delta_biomass = delta_biomass.rio.write_crs(crs)

In [ ]:
burned_clipped = burned.rio.clip(western_states.geometry)
biomass_start_clipped = (
    ds_start["predicted_biomass"].rio.write_crs(crs).rio.clip(western_states.geometry)
)
delta_biomass_clipped = delta_biomass.rio.clip(western_states.geometry)

In [ ]:
forest_area_clipped = forest_area.rio.clip(western_states.geometry)

In [ ]:
delta_biomass_burned = (
    delta_biomass_clipped.where(burned_clipped)  # .where(biomass_start_clipped > 1)
    / biomass_start_clipped
)
delta_biomass_unburned = (
    delta_biomass_clipped.where(~burned_clipped)  # .where(biomass_start_clipped > 1)
    / biomass_start_clipped
)

# Combined

In [ ]:
forest_area_ha = forest_area_clipped / 100 * 100  # convert from %gridcell to km2  # km2 to ha

print("Total forest area (million ha)")
print((forest_area_ha).sum().values / 1e6)

# Save data used for figure generation

In [ ]:
# Save data used for figure generation
import os

import numpy as np
import pandas as pd

save_dir = "figure_data/figure_3/"
os.makedirs(save_dir, exist_ok=True)

# Panel a/b data: annual time series — saved as CSV
df_timeseries = pd.DataFrame(
    {
        "year": years,
        "fire_losses_west": fire_losses_west,
        "unburned_gains_west": unburned_gains_west,
        "net_changes": net_changes,
    }
)
df_timeseries.to_csv(save_dir + "timeseries.csv", index=False)

df_usfs = pd.DataFrame(
    {
        "usfs_year": usfs_years,
        "usfs_delta": usfs_deltas,
    }
)
df_usfs.to_csv(save_dir + "usfs_timeseries.csv", index=False)

# Panel c data: spatial grids for histogram — saved as NetCDF
delta_biomass_burned.to_dataset(name="delta_biomass_burned").to_netcdf(
    save_dir + "delta_biomass_burned.nc"
)
delta_biomass_unburned.to_dataset(name="delta_biomass_unburned").to_netcdf(
    save_dir + "delta_biomass_unburned.nc"
)
forest_area_ha.to_dataset(name="forest_area_ha").to_netcdf(save_dir + "forest_area_ha.nc")

print(f"Saved all figure data to {save_dir}")